In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

np.random.seed(42)

In [2]:
# Funções de carregamento e limpeza — reutilizadas do EDA

CANDIDATE_DIRS = [Path("."), Path("data")]


def pick_path(filename: str) -> Path:
    for d in CANDIDATE_DIRS:
        p = d / filename
        if p.exists():
            return p
    return CANDIDATE_DIRS[0] / filename


FLIGHTS_PATH = pick_path("flights.csv")
AIRPORTS_PATH = pick_path("airports.csv")
AIRLINES_PATH = pick_path("airlines.csv")

assert FLIGHTS_PATH.exists(), f"Arquivo não encontrado: {FLIGHTS_PATH.resolve()}"

DELAY_CAUSE_COLS = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]


def hhmm_to_minutes(x: object) -> float:
    """Converte HHMM (ex: 5, '0005', 2354) em minutos desde 00:00.

    Retorna NaN quando não for possível converter.
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan

    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan

    s = s.zfill(4)
    if not s.isdigit() or len(s) != 4:
        return np.nan

    hh = int(s[:2])
    mm = int(s[2:])
    if not (0 <= hh <= 23 and 0 <= mm <= 59):
        return np.nan
    return hh * 60 + mm


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["scheduled_dep_min"] = df["SCHEDULED_DEPARTURE"].map(hhmm_to_minutes)
    df["scheduled_dep_hour"] = (df["scheduled_dep_min"] // 60).astype("Int64")

    df["scheduled_arr_min"] = df["SCHEDULED_ARRIVAL"].map(hhmm_to_minutes)
    df["scheduled_arr_hour"] = (df["scheduled_arr_min"] // 60).astype("Int64")
    return df


def clean_flights(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols) > 0:
        df[obj_cols] = df[obj_cols].replace({"": pd.NA})

    for c in ["CANCELLED", "DIVERTED"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("Int8")

    not_cancelled = (df["CANCELLED"] == 0) if "CANCELLED" in df.columns else pd.Series(True, index=df.index)

    present_delay_cols = [c for c in DELAY_CAUSE_COLS if c in df.columns]
    if present_delay_cols:
        df.loc[not_cancelled, present_delay_cols] = df.loc[not_cancelled, present_delay_cols].apply(
            pd.to_numeric, errors="coerce"
        ).fillna(0)

    if "CANCELLATION_REASON" in df.columns and "CANCELLED" in df.columns:
        df.loc[not_cancelled & df["CANCELLATION_REASON"].isna(), "CANCELLATION_REASON"] = "NOT_CANCELLED"
        df.loc[(~not_cancelled) & df["CANCELLATION_REASON"].isna(), "CANCELLATION_REASON"] = "UNKNOWN"

    num_like = [
        "YEAR",
        "MONTH",
        "DAY",
        "DAY_OF_WEEK",
        "DISTANCE",
        "DEPARTURE_DELAY",
        "ARRIVAL_DELAY",
        "TAXI_OUT",
        "TAXI_IN",
        "AIR_TIME",
        "ELAPSED_TIME",
        "SCHEDULED_TIME",
    ]
    for c in [x for x in num_like if x in df.columns]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if all(c in df.columns for c in ["YEAR", "MONTH", "DAY"]):
        df["date"] = pd.to_datetime(
            {"year": df["YEAR"], "month": df["MONTH"], "day": df["DAY"]}, errors="coerce"
        )

    df = add_time_features(df)
    return df


def load_sample(
    path: Path,
    chunksize: int = 250_000,
    frac_per_chunk: float = 0.03,
    max_rows: int = 400_000,
    seed: int = 42,
) -> pd.DataFrame:
    """Amostra aproximada do CSV (boa para gráficos/modelos sem estourar memória)."""
    rng = np.random.default_rng(seed)
    samples: list[pd.DataFrame] = []

    for chunk in pd.read_csv(path, chunksize=chunksize):
        s = chunk.sample(frac=frac_per_chunk, random_state=int(rng.integers(0, 2**31 - 1)))
        samples.append(s)
        if sum(len(x) for x in samples) >= max_rows:
            break

    df = pd.concat(samples, ignore_index=True)
    if len(df) > max_rows:
        df = df.sample(n=max_rows, random_state=seed).reset_index(drop=True)

    return clean_flights(df)

Clusterização (K-Means)

In [5]:
# Preparar dados para clusterização de rotas

MIN_DELAY_MINUTES = 15

if "df_ok" not in globals():
    df_ok = load_sample(
        FLIGHTS_PATH,
        frac_per_chunk=0.03,
        max_rows=300_000,
        seed=42,
    )
    # Filtrar cancelados e desvios para não poluir análises
    if "CANCELLED" in df_ok.columns:
        df_ok = df_ok[df_ok["CANCELLED"] == 0].copy()
    if "DIVERTED" in df_ok.columns:
        df_ok = df_ok[df_ok["DIVERTED"] == 0].copy()

ROUTE_FEATURES_CAT = ["AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]
ROUTE_FEATURES_NUM = ["DISTANCE", "scheduled_dep_hour"]

for c in ROUTE_FEATURES_CAT + ROUTE_FEATURES_NUM:
    if c not in df_ok.columns:
        raise ValueError(f"Coluna ausente para clusterização de rotas: {c}")

routes_df = df_ok[ROUTE_FEATURES_CAT + ROUTE_FEATURES_NUM].dropna().copy()

# garantir que categóricas sejam string
for c in ROUTE_FEATURES_CAT:
    routes_df[c] = routes_df[c].astype(str)

n_sample = min(25_000, len(routes_df))
routes_sample = routes_df.sample(n=n_sample, random_state=42)
print("Amostra para clusterização de rotas:", routes_sample.shape)

preprocess_routes = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            ROUTE_FEATURES_CAT,
        ),
        ("num", StandardScaler(), ROUTE_FEATURES_NUM),
    ]
)

X_routes = preprocess_routes.fit_transform(routes_sample)
print("Matriz de features para K-Means:", X_routes.shape)

Amostra para clusterização de rotas: (25000, 5)
Matriz de features para K-Means: (25000, 1031)


In [ ]:
# Método do cotovelo para escolher k

ks = range(2, 9)
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_routes)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.plot(list(ks), inertias, marker="o")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inércia (soma dos quadrados intra-cluster)")
ax.set_title("Método do cotovelo — clusterização de rotas")
plt.tight_layout()
plt.show()

inertias

In [ ]:
# Ajustar K-Means para um k escolhido (ex.: 5) e projetar em 2D com PCA

best_k = 5  # ajuste este valor se o "cotovelo" sugerir outro k

kmeans_routes = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_routes.fit_predict(X_routes)

pca_2d = PCA(n_components=2, random_state=42)
X_routes_pca = pca_2d.fit_transform(X_routes)

routes_plot = routes_sample.copy()
routes_plot["cluster"] = cluster_labels
routes_plot["PC1"] = X_routes_pca[:, 0]
routes_plot["PC2"] = X_routes_pca[:, 1]

print("Variância explicada pelos 2 componentes da PCA (rotas):", pca_2d.explained_variance_ratio_)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.scatterplot(
    data=routes_plot.sample(n=min(8_000, len(routes_plot)), random_state=42),
    x="PC1",
    y="PC2",
    hue="cluster",
    palette="tab10",
    alpha=0.6,
    ax=ax,
)
ax.set_title("Clusters de rotas em 2D (PCA sobre variáveis de rota)")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Resumo numérico dos clusters de rotas

cluster_summary = (
    routes_plot
    .groupby("cluster")
    .agg(
        n_voos=("cluster", "size"),
        dist_media=("DISTANCE", "mean"),
        hora_media=("scheduled_dep_hour", "mean"),
        n_companhias=("AIRLINE", "nunique"),
        n_origens=("ORIGIN_AIRPORT", "nunique"),
        n_destinos=("DESTINATION_AIRPORT", "nunique"),
    )
    .sort_values("n_voos", ascending=False)
)

cluster_summary

In [ ]:
# Distância média por cluster de rota

cluster_summary_plot = cluster_summary.reset_index()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
sns.barplot(data=cluster_summary_plot, x="cluster", y="dist_media", palette="tab10", ax=ax)
ax.set_xlabel("Cluster")
ax.set_ylabel("Distância média (milhas)")
ax.set_title("Distância média por cluster de rota")
plt.tight_layout()
plt.show()

Abordagem 2 - PCA

In [ ]:
# Preparar dados numéricos para PCA

if "MIN_DELAY_MINUTES" not in globals():
    MIN_DELAY_MINUTES = 15

if "df_ok" not in globals():
    df_ok = load_sample(
        FLIGHTS_PATH,
        frac_per_chunk=0.03,
        max_rows=300_000,
        seed=42,
    )
    # Filtrar cancelados e desvios para não poluir análises
    if "CANCELLED" in df_ok.columns:
        df_ok = df_ok[df_ok["CANCELLED"] == 0].copy()
    if "DIVERTED" in df_ok.columns:
        df_ok = df_ok[df_ok["DIVERTED"] == 0].copy()

PCA_FEATURES_NUM = ["MONTH", "DAY_OF_WEEK", "DISTANCE", "SCHEDULED_TIME", "scheduled_dep_hour"]

for c in PCA_FEATURES_NUM:
    if c not in df_ok.columns:
        raise ValueError(f"Coluna ausente para PCA: {c}")

# garantir que temos o target binário de atraso
if "atrasado" not in df_ok.columns and "ARRIVAL_DELAY" in df_ok.columns:
    df_ok["atrasado"] = (df_ok["ARRIVAL_DELAY"] > MIN_DELAY_MINUTES).astype(int)

pca_df = df_ok.dropna(subset=PCA_FEATURES_NUM + ["ARRIVAL_DELAY"]).copy()

n_sample_pca = min(30_000, len(pca_df))
pca_sample = pca_df.sample(n=n_sample_pca, random_state=42).copy()

scaler = StandardScaler()
X_pca_scaled = scaler.fit_transform(pca_sample[PCA_FEATURES_NUM])

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_pca_scaled)

pca_sample["PC1"] = X_pca[:, 0]
pca_sample["PC2"] = X_pca[:, 1]

print("Variância explicada por componente:", pca.explained_variance_ratio_)
print("Variância explicada acumulada:", pca.explained_variance_ratio_.cumsum())

In [ ]:
# Visualizar PCA em 2D, colorindo por atraso (0/1)

sample_for_plot = pca_sample.sample(n=min(15_000, len(pca_sample)), random_state=42)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

sns.scatterplot(
    data=sample_for_plot,
    x="PC1",
    y="PC2",
    hue="atrasado",
    palette={0: "tab:blue", 1: "tab:orange"},
    alpha=0.4,
    ax=ax,
)

ax.set_title("Projeção PCA do espaço de features\nCor indica atraso na chegada (> 15 min)")
plt.legend(title="Atrasado (0 = não, 1 = sim)")
plt.tight_layout()
plt.show()

## Análises

In [ ]:
# === Análises de atrasos e K-Means (aeroportos) ===

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import OneHotEncoder

MIN_DELAY_MINUTES = 15

# 1) Carregar amostra (mesma ideia do notebook completo, com amostragem por chunks)
flights_sample = load_sample(
    FLIGHTS_PATH,
    frac_per_chunk=0.03,
    max_rows=300_000,
    seed=42,
)

# 2) Manter apenas voos realizados
if "CANCELLED" in flights_sample.columns:
    flights_sample = flights_sample[flights_sample["CANCELLED"] == 0].copy()
if "DIVERTED" in flights_sample.columns:
    flights_sample = flights_sample[flights_sample["DIVERTED"] == 0].copy()

# 3) Targets/flags de atraso (>= 15 minutos)
flights_sample["is_dep_delayed_15"] = flights_sample["DEPARTURE_DELAY"] >= MIN_DELAY_MINUTES
flights_sample["is_arr_delayed_15"] = flights_sample["ARRIVAL_DELAY"] >= MIN_DELAY_MINUTES

print(f"Proporção de voos com atraso na partida >= {MIN_DELAY_MINUTES} min: {flights_sample['is_dep_delayed_15'].mean():.3f}")

# 4) Nomes de dias (se existir DAY_OF_WEEK)
if "DAY_OF_WEEK" in flights_sample.columns:
    day_name_map = {
        1: "Segunda",
        2: "Terça",
        3: "Quarta",
        4: "Quinta",
        5: "Sexta",
        6: "Sábado",
        7: "Domingo",
    }
    flights_sample["day_of_week_name"] = flights_sample["DAY_OF_WEEK"].map(day_name_map)

# 5) Faixas de horário (usando scheduled_dep_hour)
if "scheduled_dep_hour" in flights_sample.columns:
    def dep_period(hour):
        if pd.isna(hour):
            return pd.NA
        h = int(hour)
        if 0 <= h <= 5:
            return "Madrugada (0-5h)"
        if 6 <= h <= 11:
            return "Manhã (6-11h)"
        if 12 <= h <= 17:
            return "Tarde (12-17h)"
        return "Noite (18-23h)"

    flights_sample["dep_period"] = flights_sample["scheduled_dep_hour"].apply(dep_period)

# Salva taxa global para referência
baseline_rate = flights_sample["is_dep_delayed_15"].mean()


In [ ]:
# 1) Quais aeroportos são mais críticos em relação a atrasos?

min_flights = 2000

# Por aeroporto de origem (ORIGIN)
cols_required = ["ORIGIN_AIRPORT", "DEPARTURE_DELAY", "ARRIVAL_DELAY", "is_dep_delayed_15", "is_arr_delayed_15"]
missing = [c for c in cols_required if c not in flights_sample.columns]
if missing:
    raise ValueError(f"Colunas ausentes para análise de aeroportos: {missing}")

airport_delay_origin = (
    flights_sample
    .groupby("ORIGIN_AIRPORT")
    .agg(
        n_voos=("DEPARTURE_DELAY", "size"),
        pct_atraso_partida=("is_dep_delayed_15", "mean"),
        atraso_medio_partida=("DEPARTURE_DELAY", "mean"),
        pct_atraso_chegada=("is_arr_delayed_15", "mean"),
        atraso_medio_chegada=("ARRIVAL_DELAY", "mean"),
    )
    .query("n_voos >= @min_flights")
    .sort_values("pct_atraso_partida", ascending=False)
)

top_origin = airport_delay_origin.head(15)
print("Top 15 aeroportos de origem mais críticos (por % de atrasos na partida):")
display(top_origin)

# Por aeroporto de destino
if "DESTINATION_AIRPORT" in flights_sample.columns:
    airport_delay_dest = (
        flights_sample
        .groupby("DESTINATION_AIRPORT")
        .agg(
            n_voos=("DEPARTURE_DELAY", "size"),
            pct_atraso_partida=("is_dep_delayed_15", "mean"),
        )
        .query("n_voos >= @min_flights")
        .sort_values("pct_atraso_partida", ascending=False)
    )

    top_dest = airport_delay_dest.head(10)
    print("\nAeroportos de destino com maior taxa de atraso (top 10, filtrando n_voos >= 2000):")
    display(top_dest)
else:
    print("\nColuna DESTINATION_AIRPORT não disponível; pulo análise por destino.")


In [ ]:
# 2) Que características aumentam a chance de atraso?

# Companhia aérea
if "AIRLINE" in flights_sample.columns:
    airline_delay = (
        flights_sample
        .groupby("AIRLINE")
        .agg(n_voos=("DEPARTURE_DELAY", "size"), pct_atraso_partida=("is_dep_delayed_15", "mean"))
        .sort_values("pct_atraso_partida", ascending=False)
    )
    print(f"Taxa média de atrasos de partida (>= {MIN_DELAY_MINUTES} min) na amostra: {baseline_rate:.2%}")
    print("\nTaxa de atrasos por companhia aérea (top 10 mais críticas):")
    display(airline_delay.head(10))
else:
    print("Coluna AIRLINE não disponível; pulo análise por companhia.")

# Aeroporto de origem e distância (por faixas)
if "DISTANCE" in flights_sample.columns:
    distance_bins = [0, 300, 800, 1500, 3000, np.inf]
    distance_labels = ["0-300", "300-800", "800-1500", "1500-3000", ">=3000"]

    flights_sample["DISTANCE_BIN"] = pd.cut(
        flights_sample["DISTANCE"],
        bins=distance_bins,
        labels=distance_labels,
        include_lowest=True,
    )

    delay_by_distance = (
        flights_sample
        .groupby("DISTANCE_BIN")
        .agg(n_voos=("DEPARTURE_DELAY", "size"), pct_atraso_partida=("is_dep_delayed_15", "mean"))
        .sort_index()
    )

    print("\nTaxa de atrasos por faixas de distância:")
    display(delay_by_distance)
else:
    print("Coluna DISTANCE não disponível; pulo análise por distância.")


In [ ]:
# 3) Os atrasos são mais comuns em certos dias da semana ou horários?

# Dia da semana
if "DAY_OF_WEEK" in flights_sample.columns:
    day_delay = (
        flights_sample
        .groupby("DAY_OF_WEEK")
        .agg(n_voos=("DEPARTURE_DELAY", "size"), pct_atraso_partida=("is_dep_delayed_15", "mean"))
        .sort_index()
    )
    # se day_of_week_name existir, exibe
    if "day_of_week_name" in flights_sample.columns:
        names = flights_sample[["DAY_OF_WEEK", "day_of_week_name"]].drop_duplicates().set_index("DAY_OF_WEEK")
        day_delay = day_delay.join(names)
        day_delay = day_delay.rename(columns={"day_of_week_name": "day_of_week_name"})

    print("Taxa de atrasos por dia da semana:")
    display(day_delay)
else:
    print("Coluna DAY_OF_WEEK não disponível; pulo análise por dia.")

# Período do dia (bins definidos na preparação)
if "dep_period" in flights_sample.columns:
    period_delay = (
        flights_sample
        .groupby("dep_period")
        .agg(n_voos=("DEPARTURE_DELAY", "size"), pct_atraso_partida=("is_dep_delayed_15", "mean"))
        .sort_index()
    )

    print("\nTaxa de atrasos por faixa de horário da partida:")
    display(period_delay)
else:
    print("Coluna dep_period não definida; pulo análise por horário.")


In [ ]:
# 4) É possível agrupar aeroportos com perfis semelhantes? (K-Means)

min_flights = 2000

airport_delay_stats = (
    flights_sample
    .groupby("ORIGIN_AIRPORT")
    .agg(
        n_voos=("DEPARTURE_DELAY", "size"),
        pct_atraso_partida=("is_dep_delayed_15", "mean"),
        pct_atraso_chegada=("is_arr_delayed_15", "mean"),
        atraso_medio_partida=("DEPARTURE_DELAY", "mean"),
        atraso_medio_chegada=("ARRIVAL_DELAY", "mean"),
    )
    .query("n_voos >= @min_flights")
    .reset_index()
)

cluster_features = [
    "n_voos",
    "pct_atraso_partida",
    "pct_atraso_chegada",
    "atraso_medio_partida",
    "atraso_medio_chegada",
]

X_airports = airport_delay_stats[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_airports)

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

airport_clusters = airport_delay_stats.copy()
airport_clusters["cluster"] = labels

print("Distribuição de aeroportos por cluster:")
print(airport_clusters["cluster"].value_counts().sort_index())

print("\nMédias dos indicadores por cluster:")
cluster_profile = (
    airport_clusters
    .groupby("cluster")[cluster_features]
    .mean()
    .sort_index()
)

display(cluster_profile)

print("\nAeroportos por cluster (origem):")
for c in sorted(airport_clusters["cluster"].unique()):
    sub = airport_clusters[airport_clusters["cluster"] == c].sort_values("pct_atraso_partida", ascending=False)
    codes = sub["ORIGIN_AIRPORT"].astype(str).tolist()
    print(f"Cluster {c}: {', '.join(codes)}")


In [ ]:
# 5) Até que ponto conseguimos prever atrasos?

# Objetivo: prever `is_dep_delayed_15` usando apenas contexto do voo.

feature_candidates = [
    "DAY_OF_WEEK",
    "MONTH",
    "scheduled_dep_hour",
    "DISTANCE",
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
]

available_features = [c for c in feature_candidates if c in flights_sample.columns]
if "is_dep_delayed_15" not in flights_sample.columns:
    raise ValueError("A coluna is_dep_delayed_15 não existe. Refaça a preparação da célula 12.")

X = flights_sample[available_features].copy()
y = flights_sample["is_dep_delayed_15"].astype(int)

# Separa numéricas e categóricas
num_cols = [c for c in available_features if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in available_features if c not in num_cols]

X[cat_cols] = X[cat_cols].astype(str)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

baseline_rate = y.mean()
print(f"Proporção de voos atrasados (>= {MIN_DELAY_MINUTES} min): {baseline_rate:.2%}")

print("\nRelatório de classificação (atraso na partida >= 15 min):")
print(classification_report(y_test, y_pred, digits=3))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.3f}")

print("\nInterpretação (contexto básico do voo):")
print("- ROC-AUC perto de 0.5 indica pouca capacidade preditiva.")
print("- Valores intermediários (ex.: ~0.65) indicam capacidade moderada.")
print("- Valores altos indicam boa separação entre atrasados e não atrasados.")
